# 11_crash_features

## Project Title: Traffic Accident Risk Prediction (TARP)

**Unit:** SIT782  
**Prepared by:** Khalid  
**Project Team:** Suba (225094537), Burhan (224802775), Khalid (224696667)  
**Task:** Feature Engineering for Crash Data (Week 4 of 8)

This notebook creates new **time-based and road-related features** from the cleaned crash dataset. These engineered features are designed to support later stages of the project, especially pattern analysis and predictive modeling.

### Planned features
- `hour_of_day`
- `day_of_week`
- `weekend_flag`
- `rush_hour`
- `speed_zone_numeric`

### Optional supporting features
- `time_period`
- `week_part`

These features are based on the earlier EDA findings, where accident frequency was higher during commuting hours, weekdays, and major traffic corridors.


## 1. Import libraries

We start by importing the libraries required for data handling and basic validation.


In [1]:
import pandas as pd
import numpy as np

## 2. Load dataset

Let's start off by loading the data sets from previous weeks tasks. 

In [21]:
from pathlib import Path

# Define data path
data_path = Path("cleaned_crash_data.csv")

# Load dataset
df = pd.read_csv(data_path)

print("Dataset loaded from:", data_path)
df.shape
df.head()

Dataset loaded from: cleaned_crash_data.csv


,ACCIDENT_NO,ACCIDENT_DATE,ACCIDENT_TIME,ACCIDENT_TYPE,DAY_OF_WEEK,DCA_CODE,DCA_CODE_DESCRIPTION,LIGHT_CONDITION,POLICE_ATTEND,ROAD_GEOMETRY,...,PT_VEHICLE,DEG_URBAN_NAME,SRNS,RMA,DIVIDED,STAT_DIV_NAME,hour,day_name,month,year
0,T20230013207,2023-06-06,1900-01-01 22:14:00,Collision with vehicle,Tuesday,140,U TURN,Dark Street lights on,Yes,Not at intersection,...,0.0,MELB_URBAN,NaN,Arterial Other,Undivided,METRO,22.0,Tuesday,6,2023
1,T20240014902,2024-04-19,1900-01-01 02:10:00,Collision with vehicle,Friday,140,U TURN,Dark Street lights on,Yes,Not at intersection,...,0.0,MELB_URBAN,NaN,Local Road,Undivided,METRO,2.0,Friday,4,2024
2,T20160009452,2016-04-30,1900-01-01 23:50:00,Collision with vehicle,Saturday,120,HEAD ON (NOT OVERTAKING),Dark No street lights,Yes,Not at intersection,...,0.0,RURAL_VICTORIA,C,Arterial Other,Undivided,COUNTRY,23.0,Saturday,4,2016
3,T20230001223,2023-01-17,1900-01-01 15:56:00,Collision with vehicle,Tuesday,113,RIGHT NEAR (INTERSECTIONS ONLY),Day,Yes,Private property,...,0.0,RURAL_VICTORIA,A,Arterial Highway,Undivided,COUNTRY,15.0,Tuesday,1,2023
4,T20220001324,2022-01-21,1900-01-01 22:45:00,Fall from or in moving vehicle,Friday,190,FELL IN/FROM VEHICLE,Dark Street lights on,Yes,Cross intersection,...,0.0,TOWNS,NaN,Local Road,Undivided,COUNTRY,22.0,Friday,1,2022


## 3. Inspect relevant columns

Before creating features, it is good practice to confirm the columns that will be used.


In [3]:
relevant_columns = [
    "ACCIDENT_DATE", "ACCIDENT_TIME", "DAY_OF_WEEK", "SPEED_ZONE",
    "hour", "day_name", "month", "year"
]

available_columns = [col for col in relevant_columns if col in df.columns]
print("Available relevant columns:")
print(available_columns)

Available relevant columns:
['ACCIDENT_DATE', 'ACCIDENT_TIME', 'DAY_OF_WEEK', 'SPEED_ZONE', 'hour', 'day_name', 'month', 'year']


## 4. Convert and validate date column

The feature engineering in this notebook mainly uses `ACCIDENT_DATE`.  
We convert it to datetime format first.


In [4]:
df["ACCIDENT_DATE"] = pd.to_datetime(df["ACCIDENT_DATE"], errors="coerce")

print("Missing ACCIDENT_DATE values after conversion:", df["ACCIDENT_DATE"].isna().sum())
df[["ACCIDENT_DATE"]].head()

Missing ACCIDENT_DATE values after conversion: 0


,ACCIDENT_DATE
0,2023-06-06
1,2024-04-19
2,2016-04-30
3,2023-01-17
4,2022-01-21


## 5. Create engineered features

### 5.1 `hour_of_day`
Extracts the hour from the accident date or uses the existing cleaned `hour` column when available.


In [5]:
if "hour" in df.columns:
    df["hour_of_day"] = pd.to_numeric(df["hour"], errors="coerce").astype("Int64")
else:
    df["hour_of_day"] = df["ACCIDENT_DATE"].dt.hour.astype("Int64")

df[["hour_of_day"]].head()

,hour_of_day
0,22
1,2
2,23
3,15
4,22


### 5.2 `day_of_week`

Creates a weekday label such as Monday, Tuesday, and so on.

Note: The `day_of_week` feature standardizes weekday naming used in earlier EDA.

In [6]:
if "day_name" in df.columns:
    df["day_of_week"] = df["day_name"]
else:
    df["day_of_week"] = df["ACCIDENT_DATE"].dt.day_name()

df[["day_of_week"]].head()

,day_of_week
0,Tuesday
1,Friday
2,Saturday
3,Tuesday
4,Friday


### 5.3 `weekend_flag`
Creates a binary feature:
- `1` = weekend
- `0` = weekday


In [7]:
df["weekend_flag"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

df[["day_of_week", "weekend_flag"]].head()

,day_of_week,weekend_flag
0,Tuesday,0
1,Friday,0
2,Saturday,1
3,Tuesday,0
4,Friday,0


### 5.4 `rush_hour`
Creates a binary feature for peak traffic periods.

In this project, rush hour is defined as:
- **Morning peak:** 7 AM to 9 AM
- **Evening peak:** 3 PM to 6 PM


In [8]:
df["rush_hour"] = df["hour_of_day"].apply(
    lambda x: 1 if pd.notna(x) and ((7 <= x <= 9) or (15 <= x <= 18)) else 0
)

df[["hour_of_day", "rush_hour"]].head(10)

,hour_of_day,rush_hour
0,22,0
1,2,0
2,23,0
3,15,1
4,22,0
5,2,0
6,18,1
7,13,0
8,17,1
9,17,1


### 5.5 `speed_zone_numeric`
Converts the text-based speed zone into a numeric feature where possible.

Examples:
- `50 km/hr` → `50`
- `100 km/hr` → `100`

This is useful for later analysis or modeling.


In [9]:
df["speed_zone_numeric"] = (
    df["SPEED_ZONE"]
    .astype(str)
    .str.extract(r"(\d+)")
    .squeeze()
)

df["speed_zone_numeric"] = pd.to_numeric(df["speed_zone_numeric"], errors="coerce").astype("Int64")

df[["SPEED_ZONE", "speed_zone_numeric"]].head(10)

,SPEED_ZONE,speed_zone_numeric
0,50 km/hr,50
1,40 km/hr,40
2,100 km/hr,100
3,100 km/hr,100
4,50 km/hr,50
5,80 km/hr,80
6,40 km/hr,40
7,70 km/hr,70
8,40 km/hr,40
9,Not known,<NA>


## 6. Optional supporting features

These are not required in the initial plan, but they improve feature richness and may help later stages of the project.


### 6.1 `time_period`
Groups the accident hour into broader time categories.


In [10]:
def classify_time_period(hour):
    if pd.isna(hour):
        return np.nan
    hour = int(hour)
    if 7 <= hour <= 9:
        return "Morning Peak"
    elif 15 <= hour <= 18:
        return "Evening Peak"
    elif 10 <= hour <= 14:
        return "Midday"
    else:
        return "Off-Peak"

df["time_period"] = df["hour_of_day"].apply(classify_time_period)

df[["hour_of_day", "time_period"]].head(10)

,hour_of_day,time_period
0,22,Off-Peak
1,2,Off-Peak
2,23,Off-Peak
3,15,Evening Peak
4,22,Off-Peak
5,2,Off-Peak
6,18,Evening Peak
7,13,Midday
8,17,Evening Peak
9,17,Evening Peak


### 6.2 `week_part`
A simpler grouped version of weekday information.


In [11]:
df["week_part"] = df["weekend_flag"].map({0: "Weekday", 1: "Weekend"})

df[["day_of_week", "weekend_flag", "week_part"]].head()

,day_of_week,weekend_flag,week_part
0,Tuesday,0,Weekday
1,Friday,0,Weekday
2,Saturday,1,Weekend
3,Tuesday,0,Weekday
4,Friday,0,Weekday


## 7. Quick validation of new features

This section checks whether the new columns were created correctly.


In [12]:
new_feature_columns = [
    "hour_of_day", "day_of_week", "weekend_flag", "rush_hour",
    "speed_zone_numeric", "time_period", "week_part"
]

print(df[new_feature_columns].info())
df[new_feature_columns].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194352 entries, 0 to 194351
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   hour_of_day         194352 non-null  Int64 
 1   day_of_week         194352 non-null  object
 2   weekend_flag        194352 non-null  int32 
 3   rush_hour           194352 non-null  int64 
 4   speed_zone_numeric  179987 non-null  Int64 
 5   time_period         194352 non-null  object
 6   week_part           194352 non-null  object
dtypes: Int64(2), int32(1), int64(1), object(3)
memory usage: 10.0+ MB
None


,hour_of_day,day_of_week,weekend_flag,rush_hour,speed_zone_numeric,time_period,week_part
0,22,Tuesday,0,0,50,Off-Peak,Weekday
1,2,Friday,0,0,40,Off-Peak,Weekday
2,23,Saturday,1,0,100,Off-Peak,Weekend
3,15,Tuesday,0,1,100,Evening Peak,Weekday
4,22,Friday,0,0,50,Off-Peak,Weekday


## 8. Simple summaries

These quick summaries help confirm that the engineered features make sense.


In [13]:
print("Weekend flag counts:")
print(df["weekend_flag"].value_counts(dropna=False))
print("\nRush hour counts:")
print(df["rush_hour"].value_counts(dropna=False))
print("\nTime period counts:")
print(df["time_period"].value_counts(dropna=False))

Weekend flag counts:
weekend_flag
0    143798
1     50554
Name: count, dtype: int64

Rush hour counts:
rush_hour
0    103084
1     91268
Name: count, dtype: int64

Time period counts:
time_period
Evening Peak    60468
Midday          57096
Off-Peak        45988
Morning Peak    30800
Name: count, dtype: int64


## 9. Save feature-engineered dataset

This saves a new version of the dataset for later notebooks.


In [23]:
from pathlib import Path

output_path = Path("crash_features_engineered.csv")

df.to_csv(output_path, index=False)

print(f"Feature-engineered dataset saved to: {output_path.resolve()}")

Feature-engineered dataset saved to: D:\Git\SIT782\TARP01\MOP-Code\Playground\KhalidT12026\Traffic_Accident_Prediction\data\processed\crash_features_engineered.csv


## 10. Interpretation

### Feature Engineering Summary

The engineered features were validated using simple summary counts, and the results confirm that the new variables were created correctly.

#### Key Observations:

- The **weekend_flag** shows that most crashes occurred on weekdays (**143,798**) compared with weekends (**50,554**), which is consistent with earlier EDA findings.

- The **rush_hour** feature captures a substantial number of crashes during peak commuting periods, with **91,268** crashes occurring during rush hours versus **103,084** outside rush hours.

- The **time_period** feature shows that **Evening Peak (60,468)** recorded the highest number of crashes, followed by **Midday (57,096)**, while **Morning Peak (30,800)** was lower. This aligns with the earlier observation that crash frequency is highest in the late afternoon and early evening.

- The higher concentration of crashes during the evening peak compared to the morning peak suggests increased risk later in the day, potentially due to accumulated traffic congestion and driver fatigue.

#### Overall Insight:

These engineered features directly capture temporal patterns identified during exploratory data analysis, particularly peak accident periods and weekday trends. By encoding these patterns into structured variables, the dataset becomes more suitable for predictive modeling, enabling algorithms to better learn relationships between time, traffic behaviour, and accident risk.

These feature-engineered variables will be used in the next stage of the project to develop predictive models for accident risk analysis.